In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil

In [2]:
# Read the CSV and extract column names as a list
root_folder = '../../../../'
planning_data = os.path.join(root_folder,'planning')
aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')
database_location = os.path.join(aware_folder,'5_database')#'../../../5_database'

column_names_location = os.path.join(aware_folder,'1_inputs')

cols = pd.read_csv(os.path.join(column_names_location,'columns.csv'))["Column Names"].tolist()
sensors = pd.read_csv(os.path.join(planning_data,'Sensor_Mapping.csv'))
print(cols)  # Display the list of column names

daily_path = os.path.join(planning_data,'2_annual_report','pre_covid_daily_entries.csv')#'../planning_data/Annual Reports/pre_covid_daily_entries.csv'
database_path = os.path.join(database_location,'database_cleaned_p1.parquet')#'../planning_data/database/database_cleaned_p1.parquet'
daily_path_database = os.path.join(database_location,'daily.parquet')#'../planning_data/database/daily.parquet'
calendar_filename =os.path.join(database_location,'calendar.parquet')#f"../planning_data/database/calendar.parquet"

sensors

['From', 'To Time', 'Z01_T4-ChurchSt_RevDoor_IN', 'Z01_T4-ChurchSt_RevDoor_OUT', 'Z01_T4-ChurchSt_SwingDoor_IN', 'Z01_T4-ChurchSt_SwingDoor_OUT', 'Z02_T4-LibertySt_EastEsc46_IN', 'Z02_T4-LibertySt_EastEsc46_OUT', 'Z02_T4-LibertySt_WestEsc45_IN', 'Z02_T4-LibertySt_WestEsc45_OUT', 'Z02_T4-LibertySt_Stair_IN', 'Z02_T4-LibertySt_Stair_OUT', 'Z02_T4-LibertySt_Doors_IN', 'Z02_T4-LibertySt_Doors_OUT', 'Z03_T4-C1-StairEsc_StairElev_IN', 'Z03_T4-C1-StairEsc_StairElev_OUT', 'Z03_T4-C1-StairEsc_EastEsc46_IN', 'Z03_T4-C1-StairEsc_EastEsc46_OUT', 'Z03_T4-C1-StairEsc_WestEsc45_IN', 'Z03_T4-C1-StairEsc_WestEsc45_OUT', 'Z04_T4-SoConc-Entry_AllDoors_IN', 'Z04_T4-SoConc-Entry_AllDoors_OUT', 'Z05_WestProj-Street_NorthDoors_IN', 'Z05_WestProj-Street_NorthDoors_OUT', 'Z05_WestProj-Street_SouthDoors_IN', 'Z05_WestProj-Street_SouthDoors_OUT', 'Z06_WestProj-C1-StairEsc_NorthStair_IN', 'Z06_WestProj-C1-StairEsc_NorthStair_OUT', 'Z06_WestProj-C1-StairEsc_NorthEsc36_IN', 'Z06_WestProj-C1-StairEsc_NorthEsc36_OUT'

,Sensor Number,Sensor Names,Weekly Avg Mapping,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping
0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
1,2,Z01_T4-ChurchSt_RevDoor_OUT,NaN,OUT,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
2,3,Z01_T4-ChurchSt_SwingDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
3,4,Z01_T4-ChurchSt_SwingDoor_OUT,NaN,OUT,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
4,5,Z01_T4-ChurchSt_WestEsc_UP,NaN,NaN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,269,Z42_CenterPassage_Stair_OUT,NaN,OUT,NaN,C2 Main Level,NaN,Center West Passage (Level C2),Center-west Passage between Oculus and West Co...,Center-west Passage between Oculus and West Co...,42,Phase 2 Sensor,Oculus Floor
269,270,Z43_NorthWestPassage_IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
270,271,Z43_NorthWestPassage_OUT,NaN,OUT,NaN,C2 Main Level,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
271,272,Z43_SouthWestPassage_IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,Southwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN


In [3]:
df_database = (
    pd.read_parquet(database_path, engine="pyarrow")
    .drop("To Time", axis=1, errors="ignore")  # Avoid KeyError
    .assign(From=lambda df: df["From"].dt.date)  # Convert datetime to date
    .melt(id_vars=["From"], var_name="Location", value_name="Count")  # Reshape data
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")
)

df_database

,From,Location,Count,Sensor Number,Sensor Names,Weekly Avg Mapping,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping
0,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,1.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
1,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
2,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
3,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
4,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,NaN,Street Level,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102400963,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,C2 SW Oculus 1 Train Entrance,29,Phase 1 Sensor,NaN
102400964,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,C2 SW Oculus 1 Train Entrance,29,Phase 1 Sensor,NaN
102400965,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,C2 SW Oculus 1 Train Entrance,29,Phase 1 Sensor,NaN
102400966,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,184,Z29_C2_SWOculus_HubElev_INTOELEV,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,C2 SW Oculus 1 Train Entrance,29,Phase 1 Sensor,NaN


In [4]:
cols = ["From","Location","Count","Hub In/Out"]
dailys = df_database[cols]
dailys = dailys[dailys["Hub In/Out"]=="IN"]
dailys

,From,Location,Count,Hub In/Out
0,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,1.0,IN
1,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,IN
2,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,IN
3,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,IN
4,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,0.0,IN
...,...,...,...,...
102400963,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,IN
102400964,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,IN
102400965,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,IN
102400966,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,IN


In [5]:
dailys = dailys[cols]

# Ensure "Count" is numeric (convert non-numeric values to NaN)
dailys["Count"] = pd.to_numeric(dailys["Count"], errors="coerce")

# Group by date and sum counts
dailys = dailys.groupby(["From","Location"], as_index=False).agg({"Count": "sum"})

dailys

,From,Location,Count
0,2020-02-24,Z01_T4-ChurchSt_RevDoor_IN,3086.0
1,2020-02-24,Z01_T4-ChurchSt_SwingDoor_IN,670.0
2,2020-02-24,Z02_T4-LibertySt_Doors_IN,16330.0
3,2020-02-24,Z02_T4-LibertySt_EastEsc46_IN,10870.0
4,2020-02-24,Z02_T4-LibertySt_Stair_IN,4087.0
...,...,...,...
163960,2025-06-08,Z29_1Train-C2-SWOculus_AllDoors_IN,2619.0
163961,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,620.0
163962,2025-06-08,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,192.0
163963,2025-06-08,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,3431.0


In [6]:
df = pd.read_csv(daily_path)\
    .drop(["To Time"], axis = 1)\
    .melt(id_vars = "From",var_name= "Location",value_name="Count")\
    .merge(sensors, left_on = "Location",right_on = "Sensor Names")

df

cols = ["From","Location","Count","Hub In/Out"]

df = df[cols]
# dailys
df = df[df["Hub In/Out"]=="IN"]\
    .drop("Hub In/Out",axis = 1)\
    .dropna()\
    .reset_index(drop = True)
df

,From,Location,Count
0,1/1/2019,Z01_T4-ChurchSt_RevDoor_IN,3141.0
1,1/2/2019,Z01_T4-ChurchSt_RevDoor_IN,3429.0
2,1/3/2019,Z01_T4-ChurchSt_RevDoor_IN,3357.0
3,1/4/2019,Z01_T4-ChurchSt_RevDoor_IN,3779.0
4,1/5/2019,Z01_T4-ChurchSt_RevDoor_IN,2108.0
...,...,...,...
32172,2/19/2020,Z29_C2_SWOculus_HubElev_INTOELEV,418.0
32173,2/20/2020,Z29_C2_SWOculus_HubElev_INTOELEV,397.0
32174,2/21/2020,Z29_C2_SWOculus_HubElev_INTOELEV,507.0
32175,2/22/2020,Z29_C2_SWOculus_HubElev_INTOELEV,119.0


In [7]:
dailys
daily_database = pd.concat([df,dailys], axis = 0, ignore_index = True)
daily_database

,From,Location,Count
0,1/1/2019,Z01_T4-ChurchSt_RevDoor_IN,3141.0
1,1/2/2019,Z01_T4-ChurchSt_RevDoor_IN,3429.0
2,1/3/2019,Z01_T4-ChurchSt_RevDoor_IN,3357.0
3,1/4/2019,Z01_T4-ChurchSt_RevDoor_IN,3779.0
4,1/5/2019,Z01_T4-ChurchSt_RevDoor_IN,2108.0
...,...,...,...
196137,2025-06-08,Z29_1Train-C2-SWOculus_AllDoors_IN,2619.0
196138,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,620.0
196139,2025-06-08,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,192.0
196140,2025-06-08,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,3431.0


In [8]:
daily_database['From'] = pd.to_datetime(daily_database['From'])
# daily_database['Year'] = daily_database['From'].dt.year
daily_database


,From,Location,Count
0,2019-01-01,Z01_T4-ChurchSt_RevDoor_IN,3141.0
1,2019-01-02,Z01_T4-ChurchSt_RevDoor_IN,3429.0
2,2019-01-03,Z01_T4-ChurchSt_RevDoor_IN,3357.0
3,2019-01-04,Z01_T4-ChurchSt_RevDoor_IN,3779.0
4,2019-01-05,Z01_T4-ChurchSt_RevDoor_IN,2108.0
...,...,...,...
196137,2025-06-08,Z29_1Train-C2-SWOculus_AllDoors_IN,2619.0
196138,2025-06-08,Z29_C2_SWOculus_HubElev_INTOELEV,620.0
196139,2025-06-08,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,192.0
196140,2025-06-08,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,3431.0


In [9]:
# test = daily_database.groupby("From").sum("Count")
# test

# Convert 'Time_Bucket' to datetime
daily_database.to_parquet(daily_path_database,engine="pyarrow",index = False)
test_read = pd.read_parquet(daily_path_database)
print(test_read.head())
# Get the min/max dates
min_date = daily_database['From'].min()
max_date = daily_database["From"].max()

# Generate date range
date_range = pd.date_range(start=min_date, end=max_date)

# Create DataFrame
df_calendar = pd.DataFrame({
    "date_id": date_range.date,
    "week_of_year": date_range.isocalendar().week,
    "day_of_year": date_range.dayofyear,
    "day_of_week": date_range.strftime('%A'),
    "day_of_week_num": date_range.weekday,
    "month_num": date_range.month,
    "day_type": ['Weekend' if d.weekday() in [5, 6] else 'Weekday' for d in date_range],
    "month_name": date_range.strftime('%B'),
    "month_short": date_range.strftime('%b'),
    "quarter": date_range.quarter,
    "year": date_range.year,
    "year_quarter": [f"{y}-Q{q}" for y, q in zip(date_range.year, date_range.quarter)],
    "month_year": date_range.strftime('%b %Y')
}).reset_index(drop = True)
print(date_range)
df_calendar.to_parquet(calendar_filename, engine="pyarrow", index=False)

        From                    Location   Count
0 2019-01-01  Z01_T4-ChurchSt_RevDoor_IN  3141.0
1 2019-01-02  Z01_T4-ChurchSt_RevDoor_IN  3429.0
2 2019-01-03  Z01_T4-ChurchSt_RevDoor_IN  3357.0
3 2019-01-04  Z01_T4-ChurchSt_RevDoor_IN  3779.0
4 2019-01-05  Z01_T4-ChurchSt_RevDoor_IN  2108.0
DatetimeIndex(['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04',
               '2019-01-05', '2019-01-06', '2019-01-07', '2019-01-08',
               '2019-01-09', '2019-01-10',
               ...
               '2025-05-30', '2025-05-31', '2025-06-01', '2025-06-02',
               '2025-06-03', '2025-06-04', '2025-06-05', '2025-06-06',
               '2025-06-07', '2025-06-08'],
              dtype='datetime64[ns]', length=2351, freq='D')
